In [1]:
import tensorflow as tf 
from tensorflow import keras 
import tensorflow_addons as tfa 
import pandas as pd
import numpy as np 
from models import load_mf_model
from sklearn.metrics import mean_absolute_error 
#import gensim.downloader as api
#from utils import load_annotations, load_transcriptions, process_text, preprocess_text, loss_val_graph

c:\Users\SIA\anaconda3\envs\deepface-env\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


### Load data

In [2]:
AUTOTUNE = tf.data.AUTOTUNE

# Train
# scene_train_ds = tf.data.Dataset.load('./data/fullscene/train_ds/')
face_train_ds  = tf.data.Dataset.load('./data/videofaces/train_ds/')
audio_train_ds = tf.data.Dataset.load('./data/audio/train_ds/')
# text_train_ds  = tf.data.Dataset.load('./data/text/train_ds/').batch(batch_size=32)

# scene_xtrain = scene_train_ds.map(lambda x,y: x)
face_xtrain  = face_train_ds.map(lambda x,y: x)
audio_xtrain = audio_train_ds.map(lambda x,y: x)
# text__xtrain = text_train_ds.map(lambda x,y: x)
y_train      = face_train_ds.map(lambda x,y: y)

train_ds = tf.data.Dataset.zip(((face_xtrain, audio_xtrain), y_train)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)


# Valid
# scene_valid_ds = tf.data.Dataset.load('./data/fullscene/val_ds/')
face_valid_ds  = tf.data.Dataset.load('./data/videofaces/val_ds/')
audio_valid_ds = tf.data.Dataset.load('./data/audio/val_ds')
# text_valid_ds  = tf.data.Dataset.load('./data/text/val_ds/').batch(batch_size=32)

# scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xvalid  = face_valid_ds.map(lambda x,y: x)
audio_xvalid = audio_valid_ds.map(lambda x,y: x)
# text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_valid      = face_valid_ds.map(lambda x,y: y)

valid_ds = tf.data.Dataset.zip(((face_xvalid, audio_xvalid), y_valid)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)

train_ds, valid_ds

(<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

### Load model

In [3]:
mf_model = load_mf_model()
mf_model.save_weights('./weights/mf/mf_model_face_audio.t5')


### Load weights

In [4]:
mf_model.load_weights('./weights/mf/mf_model_face_audio.t5')

In [5]:
loss, mae = mf_model.evaluate(valid_ds)
(1-mae)*100

1/1 [==============================] - 9s 9s/step - loss: 0.0112 - mae: 0.0978


90.2197539806366

### Load data

In [6]:
# scene_test_ds = tf.data.experimental.load('./data/fullscene/test_ds/')
face_test_ds  = tf.data.experimental.load('./data/videofaces/test_ds/')
audio_test_ds = tf.data.experimental.load('./data/audio/test_ds')
# text_test_ds  = tf.data.experimental.load('./data/text/test_ds/').batch(batch_size=32)

# scene_xtest = scene_test_ds.map(lambda x,y: x)
face_xtest  = face_test_ds.map(lambda x,y: x)
audio_xtest = audio_test_ds.map(lambda x,y: x)
# text_xtest  = text_test_ds.map(lambda x,y: x)

y_test      = face_test_ds.map(lambda x,y: y)

test_ds = tf.data.Dataset.zip(((face_xtest, audio_xtest), y_test)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)

test_ds

Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>

In [7]:
loss, mae = mf_model.evaluate(test_ds)
(1-mae)*100

1/1 [==============================] - 0s 316ms/step - loss: 0.0143 - mae: 0.1076


89.23504576086998

In [8]:
train_ds, valid_ds

(<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

In [9]:
import datetime
t = datetime.datetime.now().strftime("%m%d_%H%M%S")

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=0)
check_point    = keras.callbacks.ModelCheckpoint(filepath='./weights/mf/mf_face_audio.t5',
                             monitor='val_mae',
                             mode='min',
                             save_best_only=True,
                             save_weights_only=True,
                             verbose=0)

optimizer = tfa.optimizers.RectifiedAdam()
mf_model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])

### Retrain model

In [10]:

history = mf_model.fit(train_ds, validation_data=valid_ds, batch_size=8, epochs=100, callbacks=[early_stopping, check_point])

Epoch 1/100
1/1 [==============================] - 49s 49s/step - loss: 0.0290 - mae: 0.1485 - val_loss: 0.0112 - val_mae: 0.0978
Epoch 2/100
1/1 [==============================] - 6s 6s/step - loss: 0.0305 - mae: 0.1497 - val_loss: 0.0112 - val_mae: 0.0978
Epoch 3/100
1/1 [==============================] - 8s 8s/step - loss: 0.0311 - mae: 0.1579 - val_loss: 0.0112 - val_mae: 0.0978
Epoch 4/100
1/1 [==============================] - 9s 9s/step - loss: 0.0327 - mae: 0.1545 - val_loss: 0.0112 - val_mae: 0.0978
Epoch 5/100
1/1 [==============================] - 10s 10s/step - loss: 0.0371 - mae: 0.1712 - val_loss: 0.0112 - val_mae: 0.0978
Epoch 6/100
1/1 [==============================] - 9s 9s/step - loss: 0.0307 - mae: 0.1626 - val_loss: 0.0111 - val_mae: 0.0971
Epoch 7/100
1/1 [==============================] - 8s 8s/step - loss: 0.0280 - mae: 0.1485 - val_loss: 0.0109 - val_mae: 0.0963
Epoch 8/100
1/1 [==============================] - 8s 8s/step - loss: 0.0251 - mae: 0.1434 - val_los

KeyboardInterrupt: 

### Load weights

In [ ]:
mf_model.load_weights('./weights/mf/mf_face_audio.t5')

## Evaluation

### Training data

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
# scene_valid_ds = tf.data.experimental.load('./data/fullscene/train_ds/')
face_train_ds  = tf.data.experimental.load('./data/videofaces/train_ds/')
audio_train_ds = tf.data.experimental.load('./data/audio/train_ds')
# text_valid_ds  = tf.data.experimental.load('./data/text/train_ds/').batch(batch_size=32)

# scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xtrain  = face_train_ds.map(lambda x,y: x)
audio_xtrain = audio_train_ds.map(lambda x,y: x)
# text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_train      = face_train_ds.map(lambda x,y: y)

train_ds = tf.data.Dataset.zip(((face_xtrain, audio_xtrain), y_train)).prefetch(buffer_size=AUTOTUNE)


In [ ]:
loss, mae = mf_model.evaluate(train_ds)

1/1 [==============================] - 1s 603ms/step - loss: 0.0117 - mae: 0.0888


In [ ]:
y_true = np.concatenate([y for x,y in train_ds], axis=0)
y_pred = mf_model.predict(train_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 1s 553ms/step


(array([92.20346 , 84.680626, 94.6655  , 92.703705, 91.340546],
       dtype=float32),
 91.11876487731934)

### Validation data

In [ ]:
# scene_valid_ds = tf.data.experimental.load('./data/fullscene/val_ds/')
face_valid_ds  = tf.data.experimental.load('./data/videofaces/val_ds/')
audio_valid_ds = tf.data.experimental.load('./data/audio/val_ds')
# text_valid_ds  = tf.data.experimental.load('./data/text/val_ds/').batch(batch_size=32)

# scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xvalid  = face_valid_ds.map(lambda x,y: x)
audio_xvalid = audio_valid_ds.map(lambda x,y: x)
# text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_valid      = face_valid_ds.map(lambda x,y: y)

valid_ds = tf.data.Dataset.zip(((face_xvalid, audio_xvalid), y_valid)).prefetch(buffer_size=AUTOTUNE)


In [ ]:
loss, mae = mf_model.evaluate(valid_ds)

1/1 [==============================] - 0s 297ms/step - loss: 0.0032 - mae: 0.0493


In [ ]:
y_true = np.concatenate([y for x,y in valid_ds], axis=0)
y_pred = mf_model.predict(valid_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 0s 282ms/step


(array([92.58926 , 94.264046, 99.92394 , 96.10797 , 92.44901 ],
       dtype=float32),
 95.06684429943562)

### Test data

In [ ]:
# scene_test_ds = tf.data.experimental.load('./data/fullscene/test_ds/')
face_test_ds  = tf.data.experimental.load('./data/videofaces/test_ds/')
audio_test_ds = tf.data.experimental.load('./data/audio/test_ds')
# text_test_ds  = tf.data.experimental.load('./data/text/test_ds/').batch(batch_size=32)


# scene_xtest = scene_test_ds.map(lambda x,y: x)
face_xtest  = face_test_ds.map(lambda x,y: x)
audio_xtest = audio_test_ds.map(lambda x,y: x)
# text_xtest  = text_test_ds.map(lambda x,y: x)

y_test      = face_test_ds.map(lambda x,y: y)

test_ds = tf.data.Dataset.zip(((face_xtest, audio_xtest), y_test)).prefetch(buffer_size=AUTOTUNE) #.shuffle(buffer_size=1000)

test_ds

<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>

In [ ]:
y_true = np.concatenate([y for x,y in test_ds], axis=0)
y_pred = mf_model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 0s 339ms/step


(array([87.3784 , 91.11008, 96.19518, 96.30198, 95.4581 ], dtype=float32),
 93.28874945640564)

In [ ]:
(1-mae)*100

array([87.3784 , 91.11008, 96.19518, 96.30198, 95.4581 ], dtype=float32)

### Save Histories

In [ ]:
import os
import pickle

os.makedirs("./histories", exist_ok=True)

with open('./histories/mf_face_audio.pkl', 'wb') as f:
    pickle.dump(history.history, f)
